# 🎯 Row-Level Optimizer — S6E4

**Idea:** Your 0.98114 submission is already very strong. Instead of rebuilding the pipeline,
we scan every row and ask: *"Do ALL other models disagree with my prediction?"*

If 8 out of 9 external models agree on a different class → high probability your prediction is wrong.

**Target:** Fix ~270 misclassified rows to reach 0.98215.

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import os

PPS = '/kaggle/input/competitions/playground-series-s6e4/'

def find_path(candidates):
    for p in candidates:
        full = p if p.endswith('/') else p + '/'
        try:
            if os.listdir(full): return full
        except: pass
    return candidates[-1] if candidates[-1].endswith('/') else candidates[-1] + '/'

PD7  = find_path(['/kaggle/input/datasets/gajananbarve/ps-s6e4-07/', '/kaggle/input/datasets/nina2025/ps-s6e4-07/'])
PD74 = find_path(['/kaggle/input/datasets/gajananbarve/ps-s6e4-74/', '/kaggle/input/datasets/nina2025/ps-s6e4-74/'])
PD85 = find_path(['/kaggle/input/datasets/gajananbarve/ps-s6e4-85/', '/kaggle/input/datasets/nina2025/ps-s6e4-85/'])

print(f"PD7  = {PD7}")
print(f"PD74 = {PD74}")
print(f"PD85 = {PD85}")

In [ ]:
# ─── Load your best submission ────────────────────────────────
# If you have it as a Kaggle dataset, change the path below
sub_path = '/kaggle/input/datasets/gajananbarve/submission-098114/submission.csv'
if not os.path.exists(sub_path):
    # Fallback: use the 0.98114 from the voting pipeline output
    sub_path = '/kaggle/working/submission_optimized.csv'
    if not os.path.exists(sub_path):
        # Rebuild it from the pipeline
        print("⚠️  0.98114 submission not found, will rebuild from pipeline")
        sub_path = None

# ─── Load ALL external models ─────────────────────────────────
external_models = {
    '7971a':  pd.read_csv(PD7  + '0.97971.a.csv')['Irrigation_Need'].values,
    '7971b':  pd.read_csv(PD7  + '0.97971.b.csv')['Irrigation_Need'].values,
    '7971c':  pd.read_csv(PD7  + '0.97971.c.csv')['Irrigation_Need'].values,
    '7971d':  pd.read_csv(PD7  + '0.97971.d.csv')['Irrigation_Need'].values,
    '7971x':  pd.read_csv(PD7  + '0.97971.x.csv')['Irrigation_Need'].values,
    '8010':   pd.read_csv(PD7  + '0.98010.csv')['Irrigation_Need'].values,
    '8057':   pd.read_csv(PD74 + '5(8) - 0.98057.csv')['Irrigation_Need'].values,
    '8074':   pd.read_csv(PD74 + '5(4) - 0.98074.csv')['Irrigation_Need'].values,
    '8072':   pd.read_csv(PD85 + '5(9) - 0.98072.csv')['Irrigation_Need'].values,
    'Aux1':   pd.read_csv(PD85 + 'Aux - 0.97254.csv')['Irrigation_Need'].values,
}

n_samples = len(next(iter(external_models.values())))
print(f"Loaded {len(external_models)} external models × {n_samples:,} rows")

if sub_path:
    base_sub = pd.read_csv(sub_path)
    base_preds = base_sub['Irrigation_Need'].values
    print(f"Loaded base submission: {sub_path}")
    print(f"Distribution: {Counter(base_preds)}")

In [ ]:
# ─── For each row, count external model agreement ─────────────
model_names = list(external_models.keys())
n_ext = len(model_names)

# Build agreement matrix
print("Analyzing row-level agreement across all external models...")

row_stats = []
for i in range(n_samples):
    votes = [external_models[m][i] for m in model_names]
    counter = Counter(votes)
    top = counter.most_common(1)[0]  # (prediction, count)
    
    base_pred = base_preds[i] if sub_path else None
    
    row_stats.append({
        'idx': i,
        'consensus': top[0],
        'consensus_count': top[1],
        'n_unique_preds': len(counter),
        'base_pred': base_pred,
        'base_agrees_with_consensus': (base_pred == top[0]) if base_pred else None,
    })

df_stats = pd.DataFrame(row_stats)

print(f"\n=== Consensus Strength ===")
print(df_stats['consensus_count'].value_counts().sort_index(ascending=False).to_string())

print(f"\n=== How many rows have unanimous external agreement (10/10)? ===")
unanimous = (df_stats['consensus_count'] == 10).sum()
print(f"  {unanimous:,} rows")

print(f"\n=== Of those unanimous rows, how many does base submission disagree with? ===")
if sub_path:
    disagree_unanimous = ((df_stats['consensus_count'] == 10) & (~df_stats['base_agrees_with_consensus'])).sum()
    print(f"  {disagree_unanimous:,} rows ← HIGH CONFIDENCE CORRECTIONS")
    
    # Show what the corrections would be
    corrections = df_stats[(df_stats['consensus_count'] == 10) & (~df_stats['base_agrees_with_consensus'])]
    if len(corrections) > 0:
        print(f"\n  Base predicted → External consensus:")
        for _, row in corrections.iterrows():
            print(f"    Row {row['idx']}: {row['base_pred']} → {row['consensus']}")

In [ ]:
# ─── Strategy 1: Correct rows where 9+/10 external models agree against base ─
print("=== Strategy 1: Correct where 9-10/10 external models agree against base ===")

corrected_preds = base_preds.copy()
n_corrected = 0
correction_details = []

thresholds = [10, 9, 8, 7]  # Try different consensus thresholds

for thresh in thresholds:
    test_preds = base_preds.copy()
    n_corr = 0
    for i in range(n_samples):
        votes = [external_models[m][i] for m in model_names]
        counter = Counter(votes)
        top = counter.most_common(1)[0]
        
        if top[1] >= thresh and base_preds[i] != top[0]:
            test_preds[i] = top[0]
            n_corr += 1
    
    print(f"  Threshold ≥{thresh}/10: {n_corr:,} corrections")
    
    # Save this version
    sub_out = pd.read_csv(PPS + 'sample_submission.csv')
    sub_out['Irrigation_Need'] = test_preds
    fname = f'submission_consensus_{thresh}.csv'
    sub_out.to_csv(fname, index=False)
    print(f"    → Saved {fname}")
    
    if thresh == 10:
        corrected_preds = test_preds
        n_corrected = n_corr

print(f"\nBaseline distribution: {Counter(base_preds)}")
print(f"Corrected (≥10/10):    {Counter(corrected_preds)}")
print(f"Rows changed: {n_corrected:,}")

In [ ]:
# ─── Strategy 2: Weighted model voting ────────────────────────
# Weight each model by its known LB score
print("=== Strategy 2: Weighted model ensemble ===")

model_weights = {
    '7971a': 0.97971,
    '7971b': 0.97971,
    '7971c': 0.97971,
    '7971d': 0.97971,
    '7971x': 0.97971,
    '8010':  0.98010,
    '8057':  0.98057,
    '8074':  0.98074,
    '8072':  0.98072,
    'Aux1':  0.97254,
}

# Normalize weights
total_w = sum(model_weights.values())
norm_weights = {k: v/total_w for k, v in model_weights.items()}

weighted_preds = []
for i in range(n_samples):
    class_scores = {'Low': 0.0, 'Medium': 0.0, 'High': 0.0}
    for m in model_names:
        pred = external_models[m][i]
        class_scores[pred] += norm_weights[m]
    weighted_preds.append(max(class_scores, key=class_scores.get))

weighted_counter = Counter(weighted_preds)
n_diff = sum(1 for a, b in zip(base_preds, weighted_preds) if a != b)
print(f"Weighted ensemble distribution: {weighted_counter}")
print(f"Differs from base in {n_diff:,} rows")

sub_out = pd.read_csv(PPS + 'sample_submission.csv')
sub_out['Irrigation_Need'] = weighted_preds
sub_out.to_csv('submission_weighted.csv', index=False)
print("  → Saved submission_weighted.csv")

In [ ]:
# ─── Strategy 3: Strong-model-only consensus (top 4 models) ──
print("=== Strategy 3: Top-4 model consensus (8074, 8072, 8057, 8010) ===")

top_models = ['8074', '8072', '8057', '8010']
top_preds_list = [external_models[m] for m in top_models]

top4_preds = []
for i in range(n_samples):
    votes = [top_preds_list[j][i] for j in range(4)]
    counter = Counter(votes)
    top = counter.most_common(1)[0]
    
    if top[1] >= 3:  # At least 3/4 agree
        top4_preds.append(top[0])
    else:
        # Use base submission as fallback
        top4_preds.append(base_preds[i])

top4_counter = Counter(top4_preds)
n_diff = sum(1 for a, b in zip(base_preds, top4_preds) if a != b)
print(f"Top-4 consensus distribution: {top4_counter}")
print(f"Differs from base in {n_diff:,} rows")

sub_out = pd.read_csv(PPS + 'sample_submission.csv')
sub_out['Irrigation_Need'] = top4_preds
sub_out.to_csv('submission_top4_consensus.csv', index=False)
print("  → Saved submission_top4_consensus.csv")

In [ ]:
# ─── Strategy 4: Hybrid — base + correct only unanimous disagreements ─
print("=== Strategy 4: Hybrid (base + unanimous external correction) ===")

hybrid_preds = base_preds.copy()
n_hybrid_corr = 0

for i in range(n_samples):
    votes = [external_models[m][i] for m in model_names]
    counter = Counter(votes)
    top = counter.most_common(1)[0]
    
    # Only correct if ALL 10 external models agree AND base disagrees
    if top[1] == 10 and base_preds[i] != top[0]:
        hybrid_preds[i] = top[0]
        n_hybrid_corr += 1

hybrid_counter = Counter(hybrid_preds)
print(f"Hybrid distribution: {hybrid_counter}")
print(f"Corrected {n_hybrid_corr:,} rows")

sub_out = pd.read_csv(PPS + 'sample_submission.csv')
sub_out['Irrigation_Need'] = hybrid_preds
sub_out.to_csv('submission_hybrid.csv', index=False)
print("  → Saved submission_hybrid.csv")

In [ ]:
# ─── Summary of all submissions ───────────────────────────────
print("\n" + "="*70)
print("SUBMISSIONS GENERATED")
print("="*70)

files = {
    '0. Base (your 0.98114)': base_preds,
    '1. Consensus ≥10/10': None,  # will load
    '2. Consensus ≥9/10': None,
    '3. Consensus ≥8/10': None,
    '4. Weighted ensemble': None,
    '5. Top-4 consensus': None,
    '6. Hybrid': None,
}

file_map = {
    '1. Consensus ≥10/10': 'submission_consensus_10.csv',
    '2. Consensus ≥9/10': 'submission_consensus_9.csv',
    '3. Consensus ≥8/10': 'submission_consensus_8.csv',
    '4. Weighted ensemble': 'submission_weighted.csv',
    '5. Top-4 consensus': 'submission_top4_consensus.csv',
    '6. Hybrid': 'submission_hybrid.csv',
}

for name in files:
    if files[name] is None and name in file_map:
        files[name] = pd.read_csv(file_map[name])['Irrigation_Need'].values

print(f"{'Submission':<30s} {'Low':>7s} {'Medium':>8s} {'High':>7s} {'Δ vs Base':>10s}")
print("-" * 70)

for name, preds in files.items():
    c = Counter(preds)
    diff = sum(1 for a, b in zip(base_preds, preds) if a != b)
    print(f"{name:<30s} {c.get('Low',0):7d} {c.get('Medium',0):8d} {c.get('High',0):7d} {diff:10d}")

print("\n🚀 Upload ALL of these to Kaggle. Even a 0.0001 improvement gets you closer.")
print("   The consensus submissions (especially ≥9/10 and ≥8/10) are most likely to help.")